In [27]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from seqeval.metrics import f1_score
import re
from collections import Counter
from tqdm import tqdm

In [28]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) 
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42) 

In [29]:
dataset = load_dataset("avramandrei/histnero")
dftrain = pd.DataFrame(dataset['train'])
dftest  = pd.DataFrame(dataset['test'])
valid   = pd.DataFrame(dataset['valid'])

In [30]:
def clean_token_label_pair(tokens, labels):
    new_toks, new_labs = [], []
    for t, l in zip(tokens, labels):
        #t_clean = t.lower()
        t_clean = t.strip()
        t_clean = re.sub(r'[^\w\s]', '', t_clean) 
        if t_clean:                                
            new_toks.append(t_clean)
            new_labs.append(l)
    return new_toks, new_labs
    
dftrain['cleaned'] = dftrain.apply(lambda r: clean_token_label_pair(r['tokens'], r['ner_tags']), axis=1)
dftrain['tokens']  = dftrain['cleaned'].apply(lambda x: x[0])
dftrain['labels']  = dftrain['cleaned'].apply(lambda x: x[1])

valid['cleaned'] = valid.apply(lambda r: clean_token_label_pair(r['tokens'], r['ner_tags']), axis=1)
valid['tokens']  = valid['cleaned'].apply(lambda x: x[0])
valid['labels']  = valid['cleaned'].apply(lambda x: x[1])

def clean_test_token(t):
    return re.sub(r'[^\w\s]', '', t.strip()) 

dftest['tokens'] = dftest['tokens'].apply(
    lambda toks: [clean_test_token(t) for t in toks if clean_test_token(t)]
)

dftrain = dftrain[dftrain['tokens'].apply(len) > 0].reset_index(drop=True)
valid   = valid[valid['tokens'].apply(len) > 0].reset_index(drop=True)

In [31]:
dftrain.head()

,id,ner_tags,tokens,doc_id,region,cleaned,labels
0,969,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[Aci, o, asceptá, o, duzina, de, junijunisiroi...",Transilvania_20-_20Gura_20Satului_201880.ann,Transylvania,"([Aci, o, asceptá, o, duzina, de, junijunisiro...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,2992,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, Asupra, cestiunii, avem, în, romaneste, oa...",Iasi_20-_20Convorbiri_20literare_201893.ann,Moldavia,"([1, Asupra, cestiunii, avem, în, romaneste, o...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,3926,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[Fe, nomenul, e, acelaşi, în, toate, emoţiunil...",Iasi_20-_20Convorbiri_20literare_201893.ann,Moldavia,"([Fe, nomenul, e, acelaşi, în, toate, emoţiuni...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,8619,"[0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[al, Bucovineĭ, substratul, formarea, desvolta...",Candela_201890.ann,Bessarabia,"([al, Bucovineĭ, substratul, formarea, desvolt...","[0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,3670,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[Este, interesant, a, vedea, luptele, şi, sili...",Iasi_20-_20Convorbiri_20literare_201893.ann,Moldavia,"([Este, interesant, a, vedea, luptele, şi, sil...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [32]:
dftest.head()

,id,ner_tags,tokens,doc_id,region
0,4319,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[Vă, invit, dar, prin, aceasta, a, vă, urma, î...",Iasi_20-_20Albina_20romaneasca_201897.ann,Moldavia
1,8346,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[Se, deşteptase, spre, un, lucru, nepotrivit, ...",Luminatorul_201924.ann,Bessarabia
2,3963,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[Cu, toată, părerea, de, reu, trebue, să, decl...",Iasi_20-_20Convorbiri_20literare_201893.ann,Moldavia
3,3151,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[Acum, îţi, închipui, poate, că, la, venirea, ...",Iasi_20-_20Convorbiri_20literare_201893.ann,Moldavia
4,3272,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[prin, întinderea, lui, asupra, fen, menelor, ...",Iasi_20-_20Convorbiri_20literare_201893.ann,Moldavia


In [33]:

word_counts = Counter()
for seq in dftrain['tokens']:
    word_counts.update(seq)

vocab = {'<PAD>': 0, '<UNK>': 1}
idx = 2
for word, cnt in word_counts.items():
    if cnt >= 1:      
        vocab[word] = idx
        idx += 1

def tokens2ids(tokens):
    return [vocab.get(t, vocab.get(t.lower(), 1)) for t in tokens]
dftrain['input_ids'] = dftrain['tokens'].apply(tokens2ids)
valid['input_ids']   = valid['tokens'].apply(tokens2ids)
dftest['input_ids']  = dftest['tokens'].apply(tokens2ids)

In [34]:
max_len = max(len(seq) for seq in dftrain['input_ids'])
max_len = 200

def pad(seq, length):
    return seq[:length] + [0] * max(0, length - len(seq))

dftrain['input_ids'] = dftrain['input_ids'].apply(lambda x: pad(x, max_len))
valid['input_ids']   = valid['input_ids'].apply(lambda x: pad(x, max_len))
dftest['input_ids']  = dftest['input_ids'].apply(lambda x: pad(x, max_len))

dftrain['labels'] = dftrain['labels'].apply(lambda x: pad(x, max_len))
valid['labels']   = valid['labels'].apply(lambda x: pad(x, max_len))

Max sequence length: 200


In [35]:
def get_casing_features(tokens, max_len):
    feats = []
    for t in tokens[:max_len]:
        if not t:
            feats.append([0,0,1])
        elif t[0].isupper() and t[1:].islower(): feats.append([1,0,0])
        elif t.isupper():                          feats.append([0,1,0])
        else:                                      feats.append([0,0,1])
    feats += [[0,0,0]] * max(0, max_len - len(feats))
    return feats

In [36]:
dftrain['casing'] = dftrain['tokens'].apply(lambda t: get_casing_features(t, max_len))
valid['casing']   = valid['tokens'].apply(lambda t: get_casing_features(t, max_len))
dftest['casing']  = dftest['tokens'].apply(lambda t: get_casing_features(t, max_len))

class NERDataset(Dataset):
    def __init__(self, input_ids, labels, casing_feats):
        self.input_ids = input_ids
        self.labels = labels
        self.casing_feats = casing_feats

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return (torch.tensor(self.input_ids[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long),
                torch.tensor(self.casing_feats[idx], dtype=torch.float32))


train_ds = NERDataset(dftrain['input_ids'].tolist(), dftrain['labels'].tolist(), dftrain['casing'].tolist())
valid_ds = NERDataset(valid['input_ids'].tolist(), valid['labels'].tolist(), valid['casing'].tolist())
test_ds  = NERDataset(dftest['input_ids'].tolist(), [[-1]*max_len]*len(dftest), dftest['casing'].tolist())

In [37]:
batch_size = 128
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    tokens, labels, casing = zip(*batch)
    tokens = pad_sequence(tokens, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-1)
    casing = torch.stack(casing)
    return tokens, labels, casing

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=128, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn = collate_fn)

In [38]:
dftrain['input_ids']

0       [2, 3, 4, 3, 5, 6, 7, 8, 9, 6, 9, 10, 11, 10, ...
1       [15, 16, 17, 18, 19, 20, 21, 22, 19, 23, 24, 2...
2       [35, 36, 37, 38, 19, 39, 40, 41, 42, 10, 43, 4...
3       [95, 96, 97, 98, 99, 100, 101, 102, 103, 6, 10...
4       [105, 106, 25, 107, 108, 82, 109, 110, 82, 111...
                              ...                        
8015    [4710, 40143, 7533, 9744, 25, 40144, 2479, 50,...
8016    [26996, 13106, 13107, 13108, 4648, 82, 22991, ...
8017    [321, 322, 80, 10956, 513, 15355, 33484, 26345...
8018    [321, 499, 700, 40156, 110, 3, 2556, 19, 730, ...
8019    [321, 6, 7464, 15172, 141, 40157, 40158, 141, ...
Name: input_ids, Length: 8020, dtype: object

In [39]:
class BiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, emb_dim=300, hidden_dim=256,  
                 num_layers=2, num_classes=11, dropout=0.3,
                 pretrained_embeddings=None):              
        super().__init__()
        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings, freeze=False, padding_idx=0)
        else:
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.lstm = nn.LSTM(emb_dim+3, hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, casing):
        emb = self.dropout(self.embedding(x))
        emb = torch.cat([emb, casing], dim=-1) 
        lstm_out, _ = self.lstm(emb)
        lstm_out = self.dropout(lstm_out)
        out = self.fc1(lstm_out)
        out = self.relu(out)
        out = self.fc2(out)
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BiLSTM_NER(
    vocab_size=len(vocab),
    emb_dim=300,                      
    hidden_dim=256,
    num_layers=2,
    dropout=0.5,
    pretrained_embeddings=None 
).to(device)

In [40]:
label_counts = Counter()
for labels in dftrain['labels']:
    for l in labels:
        if l != -1:
            label_counts[l] += 1

num_classes = 11
total_tokens = sum(label_counts.values())
weights = torch.ones(num_classes)
total = sum(label_counts[i] for i in range(1, num_classes))
weights = torch.ones(num_classes)
for i in range(1, num_classes):
    weights[i] = total / (label_counts[i] * (num_classes - 1))  
weights = weights / weights[1:].mean()
weights[0] = 0.1
weights = weights.to(device)
print("Class weights:", weights)

Class weights: tensor([0.1000, 0.3876, 0.4205, 0.4058, 0.4812, 0.3248, 1.2625, 2.5900, 1.7645,
        1.3006, 1.0624], device='cuda:0')


In [41]:
label_counts

Counter({0: 1594324,
         5: 1858,
         1: 1557,
         3: 1487,
         2: 1435,
         4: 1254,
         10: 568,
         6: 478,
         9: 464,
         8: 342,
         7: 233})

In [42]:
criterion = nn.CrossEntropyLoss(weight=weights, ignore_index=-1, label_smoothing = 0.1)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode = 'max', patience = 10)
EPOCHS = 40
best_f1 = 0.0

id2label = {
    0: 'O', 1: 'B-PERS', 2: 'I-PERS', 3: 'B-ORG', 4: 'I-ORG',
    5: 'B-LOC', 6: 'I-LOC', 7: 'B-PROD', 8: 'I-PROD',
    9: 'B-DATE', 10: 'I-DATE'
}

In [43]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for x, y, casing in progress:
        optimizer.zero_grad()
        x, y, casing = x.to(device), y.to(device), casing.to(device)
        logits = model(x, casing)
        loss = criterion(logits.view(-1, num_classes), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        progress.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}: avg loss = {avg_loss:.4f}')
    model.eval()
    true_tags, pred_tags = [], []
    with torch.no_grad():
        for x, y, casing in valid_loader:
            x, y, casing = x.to(device), y.to(device), casing.to(device)
            y = y.cpu().numpy()
            logits = model(x, casing)
            preds = logits.argmax(-1).cpu().numpy()

            for i in range(len(y)):
                true_len = (y[i] != -1).sum()
                true = [id2label[t] for t in y[i][:true_len]]
                pred = [id2label[p] for p in preds[i][:true_len]]
                true_tags.append(true)
                pred_tags.append(pred)

    f1 = f1_score(true_tags, pred_tags)
    print(f'Validation F1: {f1:.4f}')
    scheduler.step(f1)

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'best_bilstm_ner.pt')
        print('  --> Model saved')

print(f'Best validation F1: {best_f1:.4f}')

Epoch 1/40: 100%|███████████████████████████████████████████████████████████| 63/63 [00:10<00:00,  6.05it/s, loss=3.14]


Epoch 1: avg loss = 3.2778
Validation F1: 0.0000


Epoch 2/40: 100%|███████████████████████████████████████████████████████████| 63/63 [00:10<00:00,  6.26it/s, loss=3.13]


Epoch 2: avg loss = 3.1384
Validation F1: 0.0161
  --> Model saved


Epoch 3/40: 100%|███████████████████████████████████████████████████████████| 63/63 [00:09<00:00,  6.68it/s, loss=3.13]


...

Epoch 38: avg loss = 3.0588
Validation F1: 0.5011


Epoch 39/40: 100%|██████████████████████████████████████████████████████████| 63/63 [00:08<00:00,  7.85it/s, loss=3.08]


Epoch 39: avg loss = 3.0582
Validation F1: 0.4979


Epoch 40/40: 100%|██████████████████████████████████████████████████████████| 63/63 [00:08<00:00,  7.82it/s, loss=3.01]


Epoch 40: avg loss = 3.0574
Validation F1: 0.4955
Best validation F1: 0.5021


In [44]:
dataset = load_dataset("avramandrei/histnero")
dftest_with_labels = pd.DataFrame(dataset['test']) 

model.load_state_dict(torch.load('best_bilstm_ner.pt'))
model.eval()

true_tags, pred_tags = [], []

with torch.no_grad():
    for i, row in tqdm(dftest_with_labels.iterrows()):
        tokens = row['tokens']
        true_labels = row['ner_tags']
        
        input_ids = [vocab.get(t, vocab.get(t.lower(), 1)) for t in tokens]
        input_ids_padded = pad(input_ids, max_len)
        
        x = torch.tensor([input_ids_padded], dtype=torch.long).to(device)
        casing = casing = torch.tensor([get_casing_features(tokens, max_len)], dtype=torch.float32).to(device)
        logits = model(x, casing)
        preds = logits.argmax(-1).cpu().numpy()[0]
        real_len = min(len(tokens), max_len)
        
        true = [id2label[t] for t in true_labels[:real_len]]
        pred = [id2label[p] for p in preds[:real_len]]
        
        true_tags.append(true)
        pred_tags.append(pred)

test_f1 = f1_score(true_tags, pred_tags)
print(f'Test F1: {test_f1:.4f}')

C:\Users\rares\AppData\Local\Temp\ipykernel_26972\781407220.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_bilstm_ner.pt'))
1003i

Test F1: 0.3901


In [21]:
x, y = next(iter(valid_loader))

In [22]:
x

tensor([[ 2584, 12197,     3,  ...,     0,     0,     0],
        [    1,   135,  5067,  ...,     0,     0,     0],
        [   95,     6,  4352,  ...,     0,     0,     0],
        ...,
        [  127,   122,  1423,  ...,     0,     0,     0],
        [   64,     1,   122,  ...,     0,     0,     0],
        [ 8650,  1483,  1960,  ...,     0,     0,     0]])

In [86]:
pred_tags[5]

IndexError: list index out of range

In [105]:
max_len

347